In [163]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split

train_df = pd.read_csv(
    os.path.join(os.getcwd(), "data", "raw", "train.csv")
)

test_df = pd.read_csv(
    os.path.join(os.getcwd(), "data", "raw", "test.csv")
)

all_data = pd.concat([train_df, test_df], ignore_index=True)

all_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3140 entries, 0 to 3139
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Title                3139 non-null   object
 1   Content              3139 non-null   object
 2   Target Organization  3140 non-null   object
 3   Label 1              3140 non-null   object
 4   Label 2              1547 non-null   object
 5   Label 3              759 non-null    object
 6   Label 4              241 non-null    object
 7   Label 5              57 non-null     object
dtypes: object(8)
memory usage: 196.4+ KB


In [164]:
train_df['Text'] = "Title: "+ train_df['Title'] + "Content: " +train_df['Content'] + "Target Organization: " + train_df['Target Organization']
test_df['Text'] = "Title: "+ test_df['Title'] + "Content: " +test_df['Content'] + "Target Organization: " + test_df['Target Organization']


In [165]:

test_split = 0.1
train_df, eval_df = train_test_split(
    train_df,
    test_size=test_split,
)
print(f"Number of rows in training set: {len(train_df)}")
print(f"Number of rows in test set: {len(eval_df)}")


Number of rows in training set: 2483
Number of rows in test set: 276


In [166]:
non_label_cols = ['Title','Content','Target Organization','Text']

label_columns = [col for col in train_df.columns if col not in non_label_cols]

# Create a new DataFrame containing only the selected label columns
df_labels_train = train_df[label_columns]
df_labels_test = test_df[label_columns]
df_eval = eval_df[label_columns]

# Convert the label columns to lists for each row
labels_list_train = df_labels_train.values.tolist()
labels_list_test = df_labels_test.values.tolist()
labels_list_eval = df_eval.values.tolist()

In [167]:
unique_list = []
for i, label in enumerate(label_columns):
    unique_list.append(all_data[label].unique())

unique_values = list(set(list for sublist in unique_list for list in sublist))

unique_values.remove(np.nan)

In [168]:
for name in unique_values:
    train_df[name] = 0
    test_df[name] = 0
    eval_df[name] = 0

event organization
other
product launching & presentation
patent publication
investment in public company
foundation
expanding industry
department establishment
closing
subsidiary establishment
executive statement
ipo exit
service & product providing
hiring
partnerships & alliances
expanding geography
product updates
executive appointment
m&a
new initiatives & programs
company description
support & philanthropy
new initiatives or programs
funding round
clinical trial sponsorship
alliance & partnership
participation in an event
regulatory approval
article publication


In [160]:
def process_labels(df, labels_list):
    for x, label in enumerate(labels_list):
        for name in label:
            if name != np.nan:
                df.at[df.index[x], name] = 1

In [171]:
def process_labels(df, labels_list):
    labels = pd.Series(labels_list, index=df.index)
    dummies = (
        labels.explode()
        .dropna()
        .pipe(pd.get_dummies)
        .groupby(level=0)
        .max()
        .astype(int)
    )
    df[dummies.columns] = dummies
    return df

In [172]:
train_df = process_labels(train_df, labels_list_train)
test_df = process_labels(test_df, labels_list_test)
eval_df = process_labels(eval_df, labels_list_eval)

In [174]:
for df in [train_df, test_df, eval_df]:
    df["labels"] = df[label_columns].values.tolist()

In [178]:
train_df['labels'] = train_df[unique_values].values.tolist()

In [179]:
train_df['labels']

317     [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1235    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, ...
1402    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...
2715    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...
234     [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
                              ...                        
2272    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...
361     [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1202    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...
1589    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
8       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
Name: labels, Length: 2483, dtype: object

In [7]:
mapping_values = {}
for i in range(len(unique_values)):
    mapping_values.update({unique_values[i]: i})



In [8]:
mapping_values

{'event organization': 0,
 'other': 1,
 'product launching & presentation': 2,
 'patent publication': 3,
 'investment in public company': 4,
 'foundation': 5,
 'expanding industry': 6,
 'department establishment': 7,
 'closing': 8,
 'subsidiary establishment': 9,
 'executive statement': 10,
 'ipo exit': 11,
 'service & product providing': 12,
 'hiring': 13,
 'partnerships & alliances': 14,
 'expanding geography': 15,
 'product updates': 16,
 'executive appointment': 17,
 'm&a': 18,
 'new initiatives & programs': 19,
 nan: 20,
 'company description': 21,
 'support & philanthropy': 22,
 'new initiatives or programs': 23,
 'funding round': 24,
 'clinical trial sponsorship': 25,
 'alliance & partnership': 26,
 'participation in an event': 27,
 'regulatory approval': 28,
 'article publication': 29}

In [9]:
for label in label_columns:
    train_df[label] = train_df[label].map(mapping_values)
    test_df[label] = test_df[label].map(mapping_values)
    eval_df[label] = eval_df[label].map(mapping_values)

In [10]:
test_df[100:110]

,Title,Content,Target Organization,Label 1,Label 2,Label 3,Label 4,Label 5,Text
100,Prepaid AC lounge inaugurated at VZM railway s...,Vizianagaram: Divisional railway manager (DRM)...,Light Lounge,1,20,20,20,20,Title: Prepaid AC lounge inaugurated at VZM ra...
101,Nurses: Staffing levels thin at Los Robles,View Comments\nView Comments\nNurses negotiati...,Nurse Staffing,1,20,20,20,20,Title: Nurses: Staffing levels thin at Los Rob...
102,Apartment fire in Willmar sends one to the hos...,"10:29 am, Nov. 21, 2021\n\nThe Willmar Fire De...",Carris Health,1,20,20,20,20,Title: Apartment fire in Willmar sends one to ...
103,VouchForMe (IPL) Price Hits $0.0012,"Get Rating\n) by 3.6% in the 4th quarter, Hold...","SxanPro, LLC",1,20,20,20,20,Title: VouchForMe (IPL) Price Hits $0.0012Cont...
104,Why Amazon makes you click a box to redeem cou...,Attention holiday shoppers: Buy now before it ...,box-planner,1,20,20,20,20,Title: Why Amazon makes you click a box to red...
105,Super-Earths: Long-lasting radiation shields m...,The laser system at the National Ignition Faci...,Rayshield,1,20,20,20,20,Title: Super-Earths: Long-lasting radiation sh...
106,FTC Reaches Settlement With Flo Health Over Fe...,"John.McKinnon@wsj.com\nUpdated Jan. 14, 2021 7...",Flo Healthcare,1,20,20,20,20,Title: FTC Reaches Settlement With Flo Health ...
107,How to increase weapon accuracy using Octane's...,"At present, Octane is the most picked characte...",ApexHealth,1,20,20,20,20,Title: How to increase weapon accuracy using O...
108,Is increased panting cause for alarm?,"By Dr. John De Jong| Ask the Vet\nMarch 27, 20...",Coyotebio-Lab,1,20,20,20,20,Title: Is increased panting cause for alarm?Co...
109,Pivot Point Consulting Issues 2022 Healthcare ...,approximately 20% of hospitals and health clin...,IT&Care,1,20,20,20,20,Title: Pivot Point Consulting Issues 2022 Heal...


from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

# Example usage
text = "DeBERTa-small what tokenizer to use with this model"
tokens = tokenizer(text)
print(tokens)

In [11]:
from datasets import load_dataset, DatasetDict

dataset_dict = DatasetDict({
    "train": train_df,
    "test": test_df,
    "eval": eval_df})

In [17]:
dataset_dict['eval'][100:110]

,Title,Content,Target Organization,Label 1,Label 2,Label 3,Label 4,Label 5,Text
1651,Ambition is a Growth Enabler! Heres How to Lev...,11:15 AM ET\nFont Size:\nSome might think the ...,Este Medical Group,15,21,10,20,20,Title: Ambition is a Growth Enabler! Heres How...
2411,TrustedReviews Limited launches Trusted Review...,Share:\nShare\nTrustedReviews Limited has laun...,Liquiproof LABS,23,26,10,21,20,Title: TrustedReviews Limited launches Trusted...
240,Domestic Services jobs,#\nMeridian health are recruiting for experien...,StaffBank Recruitment,1,20,20,20,20,Title: Domestic Services jobsContent: #\nMerid...
362,"CareLinc Medical Equipment, CareLinc, West Mic...","Apr 21, 2021 / 01:38 PM EDT\n/\nApr 21, 2021 /...",Carelinc,1,20,20,20,20,"Title: CareLinc Medical Equipment, CareLinc, W..."
1306,Sony Honda forge strategic e-mobility alliance,Read time\n2min 20sec\nJapanese multinational ...,AllianceChicago,9,26,2,10,20,Title: Sony Honda forge strategic e-mobility a...
2195,Wasabi Technologies Becomes Official Cloud Sto...,"Wasabi Technologies\n, the hot cloud storage c...",ALung Technologies,26,10,21,20,20,Title: Wasabi Technologies Becomes Official Cl...
648,Fourth Quarter Looks Promising; Input Cost Inc...,"Jan 27, 2022, 05:31 PM\nIST (Published)\nMini\...",Symphony Corporation,10,20,20,20,20,Title: Fourth Quarter Looks Promising; Input C...
2287,Deadline looms for North East Business Awards ...,privacy notice\nThe deadline for the North Eas...,Sage Dental,10,21,20,20,20,Title: Deadline looms for North East Business ...
2484,Afghanistan army veteran turned Amazon manager...,"Published: 06:00, 24 September 2021\nMore news...",Invictus Games,10,20,20,20,20,Title: Afghanistan army veteran turned Amazon ...
1727,ChemDirect Partnership,"Posted on: October 22nd, 2019\nTedia is please...",ChemDirect,26,20,20,20,20,Title: ChemDirect PartnershipContent: Posted o...


In [18]:
!pip install Pathlib


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
from transformers import DistilBertTokenizer
from Pathlib import Path
import yaml

ROOT = Path(__file__).resolve().parent.parent
config_path = ROOT / "configs/configs.yaml"

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)


tokenizer = DistilBertTokenizer.from_pretrained(cfg["model"]["name"], do_lower_case=True)
# Tokenization function
def preprocess_function(examples):
    return tokenizer(
        examples[cfg["data"]["text_column"]],
        truncation=True,
        padding="max_length",
        max_length=cfg["model"]["max_length"],
    )

# Apply tokenization to all splits
encoded_with_text = dataset_dict.map(preprocess_function, batched=True, desc="Tokenizing")

print(encoded_with_text["train"][0])

ModuleNotFoundError: No module named 'Pathlib'

In [20]:
print(train_df["Text"].isna().sum())

2


In [ ]:
train_texts = train_df['Text'].tolist()
train_labels = labels_list_train

eval_texts = test_df['Text'].tolist()
eval_labels = labels_list_test

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

train_encodings = tokenizer(train_texts, padding="max_length", truncation=True, max_length=512)
eval_encodings = tokenizer(eval_texts, padding="max_length", truncation=True, max_length=512)


/Users/conorcremin/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece installed to convert a slow tokenizer to a fast one.

In [22]:
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
config_path = ROOT / "config/config.yaml"


NameError: name '__file__' is not defined

In [31]:
import os

root = os.getcwd()
print(root)

/Users/conorcremin/Repo/Legal Document Classifier


In [33]:
from pathlib import Path
import yaml 
import os

root = os.getcwd()

#ROOT = Path(__file__).resolve().parent.parent
config_path = Path(root + "/" + str(config_path) ).resolve()
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

In [43]:
from datasets import load_from_disk
from pathlib import Path

processed_path = (Path(root) / cfg["data"]["processed_path"]).resolve()

dataset = load_from_disk(f"file://{processed_path}")

tokenized_train_dataset = dataset["train"]
print(tokenized_train_dataset)

TypeError: must be called with a dataclass type or instance